# Pipeline — Patents, Papers, Parts & Planet

This single notebook runs the full data pipeline from ingestion to website export.
Each cell runs one script from the `scripts/` directory and shows its output.

You can also run any step on its own from the terminal:
```bash
python scripts/01_ingest_papers.py
```

## Steps
| Step | Script | What it does |
|------|--------|--------------|
| 1 | `01_ingest_papers.py` | Download papers from OpenAlex |
| 2 | `02_ingest_patents.py` | Download patents from Lens.org |
| 3 | `03_ingest_projects.py` | Load iGEM projects and parts from CSV |
| 4 | `04_embed.py` | Generate sentence embeddings |
| 5 | `05_cluster.py` | UMAP projection + HDBSCAN clustering |
| 6 | `06_visualize.py` | Export JSON files for the website |

After step 6, a **Case Study section** below explores the carbon-capture subset.

---
**Before running:**
- Copy `.env.example` to `.env` and fill in your API keys
- Place iGEM CSV files in `data/raw/projects/` and `data/raw/parts/`
- Install dependencies: `pip install -r requirements.txt`

In [7]:
import sys
import os
from pathlib import Path

# Add repo root to path so scripts can import src/
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

# Ensure CWD is the notebooks/ directory so that %run ../scripts/... always works.
# On Python 3.12 + macOS, %run with a relative path calls os.getcwd() internally
# and will fail with FileNotFoundError if the kernel started from a different directory.
os.chdir(REPO_ROOT / 'notebooks')

print(f'Working from: {REPO_ROOT}')
print(f'CWD set to:   {Path.cwd()}')

Working from: /Users/zakh/Documents/CSH Thesis/Project Iterations/Patents Papers Parts and Planet
CWD set to:   /Users/zakh/Documents/CSH Thesis/Project Iterations/Patents Papers Parts and Planet/notebooks


## Step 1 — Ingest Papers

Builds the synthetic biology paper corpus in three layers:

1. **Core keywords** — `'synthetic biology'`, `'synthetic genomics'`, `'synthetic genome'`
   High-precision retrieval; these terms almost always mean synthetic biology.
2. **Subfield keywords** — `'BioBrick'`, `'repressilator'`, `'minimal genome'`, etc.
   Catches foundational work that predates or avoids the 'synthetic biology' label.
3. **Citation expansion** — backward and forward snowballing from Layer 1 seed papers.
   Captures related work using different terminology.

Each paper is tagged with a `retrieval_reason` field recording which layer found it.
Broad terms like `'metabolic engineering'` are **not** used for retrieval — they would
swamp the corpus. They appear only as post-hoc classification labels.

Strategy: Shapira, Kwon & Youtie (2017, *Scientometrics* 112(3), 1457–1478).

In [ ]:
%run ../scripts/01_ingest_papers.py

## Step 2 — Ingest Patents

Retrieves synthetic biology patents from Lens.org using a **combined IPC classification
+ keyword strategy**, following van Doren, Koenigstein & Reiss (2013).

### Why keywords alone are not enough

Synthetic biology has no dedicated IPC or CPC patent classification code — patents are
scattered across C12N (genetic engineering), C12P (fermentation), C12Q (diagnostic
nucleic acids), C12S (processes using enzymes), and C40B (combinatorial chemistry /
gene libraries). A pure keyword search on terms like "synthetic biology" dramatically
overestimates activity because the field's terminology overlaps heavily with general
biotechnology and genetic engineering (Oldham & Hall, 2018).

### Strategy: IPC codes + keywords (AND logic)

Van Doren et al. (2013) retrieved 1,195 patents (1990–2010) using a three-part approach:

1. **IPC scope filter** — restrict to the relevant technical classes:
   `C12N`, `C12P`, `C12Q`, `C12S`, `C40B`
2. **Knowledge & engineering keywords** — `"synthetic biology"`, `"genetic circuit"`,
   `"synthetic genome"`, `"DNA assembly"`, `"BioBrick"`
3. **Enabling technology keywords** — `"gene synthesis"`, `"DNA synthesis"`,
   `"cell-free"`, `"metabolic pathway engineering"`

Combining the IPC filter (AND) with keywords means we only retrieve patents that are
both in the right technical class *and* mention synthetic biology concepts — reducing
false positives from general biotech.

### References
- van Doren, D., Koenigstein, S., & Reiss, T. (2013). The development of synthetic
  biology: a patent analysis. *Systems and Synthetic Biology*, 7(4), 209–220.
  https://doi.org/10.1007/s11693-013-9121-7
- Oldham, P., & Hall, S. (2018). Synthetic Biology: Mapping the Patent Landscape.
  *bioRxiv*. https://doi.org/10.1101/483826

In [8]:
%run ../scripts/02_ingest_patents.py

FileNotFoundError: [Errno 2] No such file or directory

## Step 3 — Ingest iGEM Projects and Parts

Loads local CSV files downloaded from the iGEM Registry and saves
`data/processed/projects.csv` and `data/processed/parts.csv`.

Place iGEM data in:
- `data/raw/projects/igem_projects.csv`
- `data/raw/parts/igem_parts.csv`

In [4]:
%run ../scripts/03_ingest_projects.py

=== Step 3: Ingest iGEM Projects and Parts ===

Projects: 4615, carbon capture: 141
Skipping parts — run scripts/03b_fetch_parts.py to generate it.
Saved projects.csv to data/processed/
No parts data — parts.csv not written. Run scripts/03b_fetch_parts.py to add parts.


In [3]:
%run ../scripts/03b_fetch_parts.py

INFO  Loading parts from cache: /Users/zakh/Documents/CSH Thesis/Project Iterations/Patents Papers Parts and Planet/data/raw/parts/parts_cache.jsonl
INFO  Loaded 500 parts from cache


=== Step 3b: Fetch iGEM Parts from Registry API ===

Loaded 4615 projects for team matching.
Loaded 8720 papers for DOI matching.

Phase 1: Fetch all parts from Registry API
  500 parts loaded.

Phase 2: Fetch authors (part → org UUID)
  Fetching authors for 50 parts (450 already cached)…
  This phase takes ~2 hours for the full corpus. Safe to interrupt.
  Done. Authors cached for 500 parts.

Phase 3: Fetch organisations (org UUID → team name)
  Fetching 53 organisations (10 already cached)…


INFO  Parts matched to a project team: 500
INFO  Parts without a team match:      0


  Done. 63 orgs cached.

Phase 4: Build outputs
  Fetching composition for 500 matched parts…
  100/500 done


WARNING  Request failed for https://api.registry.igem.org/v1/parts/7f9c6b20-aa6e-4a08-9b48-3ef22cd247b3/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part 7f9c6b20-aa6e-4a08-9b48-3ef22cd247b3


  200/500 done


WARNING  Request failed for https://api.registry.igem.org/v1/parts/bbc47dde-d989-4fcb-8a76-96c958730d24/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part bbc47dde-d989-4fcb-8a76-96c958730d24
WARNING  Request failed for https://api.registry.igem.org/v1/parts/ad99e85a-2f44-49f1-a4f8-312115076644/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part ad99e85a-2f44-49f1-a4f8-312115076644
WARNING  Request failed for https://api.registry.igem.org/v1/parts/63b13e1a-a5c3-41d6-a572-9ebfe7ed8138/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part 63b13e1a-a5c3-41d6-a572-9ebfe7ed8138
WARNING  Request failed for https://api.registry.igem.org/v1/parts/8a6c538d-cec0-409d-bea7-74ea21782061/composition: ('Co

  300/500 done


WARNING  Request failed for https://api.registry.igem.org/v1/parts/f63b2e6d-1cf1-4543-b624-f8c9bfbac8cb/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part f63b2e6d-1cf1-4543-b624-f8c9bfbac8cb
WARNING  Request failed for https://api.registry.igem.org/v1/parts/b35e1c14-216f-4477-869a-d01ec326ad45/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part b35e1c14-216f-4477-869a-d01ec326ad45
WARNING  Request failed for https://api.registry.igem.org/v1/parts/487618eb-8b24-4644-b1ad-b5ee34a8c771/composition: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
WARNING  Could not fetch composition for part 487618eb-8b24-4644-b1ad-b5ee34a8c771
WARNING  Request failed for https://api.registry.igem.org/v1/parts/2d0790a8-c6b7-425d-9189-915d647b5a87/composition: ('Co

  400/500 done
  500/500 done
  Done. 574 composition edges found.

Building part → paper edges via DOI matching…

Results:
  Parts total:               500
  Parts with location:       488
  Parts with source DOIs:    84
  Team edges:                500
  Composition edges:         574
  Part→paper DOI edges:      85 (0 matched to corpus)

Saved:
  data/processed/parts.csv
  data/processed/part_team_edges.csv
  data/processed/part_composition_edges.csv
  data/processed/part_paper_edges.csv

Done.


## Step 4 — Generate Embeddings

Encodes every artifact's `text` field as a 768-dimensional dense vector using
**SPECTER** (`allenai-specter`), a transformer model trained on scientific papers
using citation-informed contrastive learning. SPECTER is better suited than
general-purpose sentence models for capturing semantic relatedness across patents,
papers, and student projects in scientific domains like synthetic biology. It loads
cleanly via `sentence-transformers` with no extra dependencies.

Reference: Cohan et al. (2020) "SPECTER: Document-level Representation Learning
using Citation-informed Transformers." ACL 2020. https://arxiv.org/abs/2004.13313

Results are cached in `data/embeddings/embeddings.json`. Already-encoded items
are skipped, so this is safe to re-run.

> **Note:** The first run downloads ~440 MB of model weights. If you switch
> embedding models, delete `data/embeddings/embeddings.json` first so the cache
> is rebuilt — vectors from different models are not compatible.

In [1]:
%run ../scripts/04_embed.py

=== Step 4: Generate Embeddings ===

Loaded 9466 records from papers.csv
Empty: patents.csv (skipping)
Loaded 4615 records from projects.csv

Total: 14081 artifacts across 2 types

Loading model: allenai-specter
(First run downloads ~440 MB of model weights — this is normal.)
Model loaded.


Embedding: 100%|██████████| 32/32 [06:24<00:00, 12.02s/it]


Embedding cache has 14081 entries.
Saved to data/embeddings/embeddings.json


## Step 5 — UMAP Projection and Clustering

Projects the high-dimensional embeddings to 2D with UMAP, then finds thematic
clusters with HDBSCAN. Noise points (no cluster found) get label `-1`.

**What gets clustered:** papers, patents, and iGEM projects — not parts.
Parts are short registry entries without proper abstracts, so they are not embedded.
The iGEM projects that contain each part are embedded instead.

**Two-stage UMAP pipeline** (following Grootendorst 2022, BERTopic):
- Stage 1: reduce to N dimensions (e.g. 10–20D) → feed to HDBSCAN. Clustering on 2D alone distorts structure.
- Stage 2: reduce to 2D separately → used only for the plot below.

**Adjust the parameters in the next cell, then run all three cells to see the effect.**

| Parameter | What it does |
|---|---|
| `UMAP_N_COMPONENTS_CLUSTER` | Dimensions used for clustering. More = richer structure, slower. Try 5–20. |
| `UMAP_N_NEIGHBORS` | How many nearby points UMAP considers. Smaller → tight local clusters; larger → broader global shape. |
| `UMAP_MIN_DIST` | How compressed points are in 2D. Smaller → tighter clumps; larger → more spread out. |
| `HDBSCAN_MIN_CLUSTER_SIZE` | Smallest group that counts as a cluster. Larger → fewer, bigger clusters. |
| `HDBSCAN_MIN_SAMPLES` | Controls noise sensitivity. Larger → more points labeled as noise (-1). |

In [23]:
# ── Step 5 parameters ──────────────────────────────────────────────────────────
# Edit these values, then run this cell and the two below it.
# These are written into config/settings.yaml before the script runs.

# UMAP stage 1 — high-dimensional reduction used for clustering
UMAP_N_COMPONENTS_CLUSTER = 50   # try: 5 (faster), 15–20 (richer structure)

# UMAP stage 2 — 2D reduction used only for the plot below
UMAP_N_COMPONENTS_VIZ = 2        # keep at 2 for a 2D scatter

# UMAP shared settings
UMAP_N_NEIGHBORS = 20    # try: 5 (tight local), 30 (broad global), 50 (very smooth)
UMAP_MIN_DIST    = 0.5   # try: 0.0 (dense clumps), 0.5 (spread), 1.0 (uniform)

# HDBSCAN — clusters on the high-D output, not the 2D plot
HDBSCAN_MIN_CLUSTER_SIZE = 20   # try: 5 (many small), 50 (few large), 100 (very coarse)
HDBSCAN_MIN_SAMPLES      = 5    # try: 1 (less noise), 20 (stricter, more noise)

# --- Write parameters into config/settings.yaml so 05_cluster.py picks them up ---
import yaml
_cfg_path = REPO_ROOT / 'config' / 'settings.yaml'
with open(_cfg_path) as _f:
    _cfg = yaml.safe_load(_f)

_cfg['umap']['n_components_cluster']     = UMAP_N_COMPONENTS_CLUSTER
_cfg['umap']['n_components_viz']         = UMAP_N_COMPONENTS_VIZ
_cfg['umap']['n_neighbors']              = UMAP_N_NEIGHBORS
_cfg['umap']['min_dist']                 = UMAP_MIN_DIST
_cfg['clustering']['min_cluster_size']   = HDBSCAN_MIN_CLUSTER_SIZE
_cfg['clustering']['min_samples']        = HDBSCAN_MIN_SAMPLES

with open(_cfg_path, 'w') as _f:
    yaml.dump(_cfg, _f, default_flow_style=False, sort_keys=False)

print("Config updated:")
print(f"  UMAP  cluster dims={UMAP_N_COMPONENTS_CLUSTER}, viz dims={UMAP_N_COMPONENTS_VIZ}")
print(f"        n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST}")
print(f"  HDBSCAN  min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}")

Config updated:
  UMAP  cluster dims=50, viz dims=2
        n_neighbors=20, min_dist=0.5
  HDBSCAN  min_cluster_size=20, min_samples=5


In [ ]:
%run ../scripts/05_cluster.py
%run ../scripts/05c_plot.py

=== Step 5: UMAP Projection and HDBSCAN Clustering ===

13335 artifacts, 13335 cached embeddings
Embedding matrix: (13335, 768)

Stage 1 — UMAP to 50D for clustering (n_neighbors=20, min_dist=0.5)...


/Users/zakh/Documents/CSH Thesis/Project Iterations/Patents Papers Parts and Planet/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP output: (13335, 50)

Stage 2 — UMAP to 2D for visualization...


/Users/zakh/Documents/CSH Thesis/Project Iterations/Patents Papers Parts and Planet/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP output: (13335, 2)

Running HDBSCAN on 50D coords (min_cluster_size=20)...
Found 2 clusters, 0 noise points
0      145
1    13190

Saved projections.json and all_artifacts.csv
Saved coords_nd.npy (13335, 50) and valid_ids.json (13335 ids)


## Step 5b — Label Clusters with Claude Haiku

Uses Claude Haiku to assign a short topic label to each cluster (e.g. "CRISPR gene editing tools",
"cell-free protein synthesis"). Results are cached — re-running only calls the API for new or missing clusters.

**How it works:**
1. For each cluster, the 20 titles closest to the cluster's centroid in high-dimensional space are selected.
2. Batches of 10 clusters are sent to Claude Haiku in a single API call.
3. Labels are written to `cluster_labels.json` and merged back into `all_artifacts.csv` and `projections.json`.

If `subclustering.enabled` is set to `true` in `config/settings.yaml`, large clusters (≥ 100 points) are
also split into sub-clusters and each sub-cluster is labeled separately.

> **After re-running Step 5:** delete `data/embeddings/cluster_labels.json` before running this step,
> because HDBSCAN cluster integers may have renumbered.

In [ ]:
# ── Step 5b parameters ─────────────────────────────────────────────────────────
# Edit these, then run this cell and the two below it.

LABEL_SAMPLE_SIZE = 20   # how many titles per cluster to send to the LLM
                         # more = better labels, slightly higher API cost
LABEL_BATCH_SIZE  = 10   # clusters per API call (10 is a good default)

# Write into settings.yaml
import yaml
_cfg_path = REPO_ROOT / 'config' / 'settings.yaml'
with open(_cfg_path) as _f:
    _cfg = yaml.safe_load(_f)

_cfg.setdefault('labeling', {})['sample_size'] = LABEL_SAMPLE_SIZE
_cfg.setdefault('labeling', {})['batch_size']  = LABEL_BATCH_SIZE

with open(_cfg_path, 'w') as _f:
    yaml.dump(_cfg, _f, default_flow_style=False, sort_keys=False)

print(f"Labeling: sample_size={LABEL_SAMPLE_SIZE}, batch_size={LABEL_BATCH_SIZE}")
print("To enable sub-clustering, set subclustering.enabled: true in config/settings.yaml")

In [ ]:
%run ../scripts/05b_label_clusters.py

In [ ]:
# ── Step 5b preview ────────────────────────────────────────────────────────────
# Shows a random sample of 25 cluster labels so you can judge quality.

import json, random
_labels_path = REPO_ROOT / 'data' / 'embeddings' / 'cluster_labels.json'

if not _labels_path.exists():
    print("cluster_labels.json not found — run the cell above first.")
else:
    with open(_labels_path) as _f:
        _labels = json.load(_f)

    # Count artifacts per cluster for context
    import pandas as pd
    _df = pd.read_csv(REPO_ROOT / 'data' / 'processed' / 'all_artifacts.csv')
    _sizes = _df[_df['cluster_label'] >= 0]['cluster_label'].value_counts()

    print(f"{len(_labels)} clusters labeled\n")
    print(f"{'Cluster':>8}  {'Size':>5}  Label")
    print("-" * 50)

    _sample_ids = random.sample(list(_labels.keys()), min(25, len(_labels)))
    for cid in sorted(_sample_ids, key=int):
        size = _sizes.get(int(cid), 0)
        print(f"{cid:>8}  {size:>5}  {_labels[cid]}")

    # Show sub-cluster labels if present
    _sub_path = REPO_ROOT / 'data' / 'embeddings' / 'subcluster_labels.json'
    if _sub_path.exists():
        with open(_sub_path) as _f:
            _sub = json.load(_f)
        n_sub = sum(1 for k in _sub if not k.startswith('_'))
        print(f"\n{n_sub} sub-cluster labels also available.")
        _sub_sample = {k: v for k, v in _sub.items() if not k.startswith('_')}
        for k in list(_sub_sample)[:10]:
            print(f"  {k}: {_sub_sample[k]}")

## Step 6 — Export for Website

Writes the three JSON files that the website's interactive visualizations load:
- `artifacts.json` — metadata for every artifact
- `projections.json` — UMAP coordinates
- `cities.json` — city-level artifact counts

In [7]:
%run ../scripts/06_visualize.py

=== Step 6: Export Data for Website ===

14064 artifacts loaded
Wrote artifacts.json (14064 records)
Copied projections.json
Wrote cities.json (2775 cities)

All files written to website/assets/data/


---

# Case Study — Carbon Capture in Synthetic Biology

This section explores the carbon-capture subset of the corpus in detail.
It is designed to be readable as a standalone reproducibility companion on the website.

**What we look at:**
1. How many artifacts were tagged as carbon-capture, and of what types?
2. Where do carbon-capture artifacts sit in the shared semantic space?
3. How has carbon-capture activity changed over time?
4. Which cities show the most carbon-capture activity?

In [ ]:
import json
import pandas as pd
import plotly.express as px
from pathlib import Path

PROCESSED   = REPO_ROOT / 'data' / 'processed'
EMBEDDINGS  = REPO_ROOT / 'data' / 'embeddings'

In [ ]:
# --- Load all processed datasets ---

dfs = {}
for name in ['papers', 'patents', 'projects', 'parts']:
    path = PROCESSED / f'{name}.csv'
    if path.exists():
        dfs[name] = pd.read_csv(path)
        print(f'{name}: {len(dfs[name])} records')

all_artifacts = pd.concat(list(dfs.values()), ignore_index=True)
print(f'\nTotal: {len(all_artifacts)} artifacts')

In [ ]:
# --- Filter to carbon-capture subset ---

cc = all_artifacts[all_artifacts['case_study_flag'] == True].copy()

print(f'Carbon capture artifacts: {len(cc)}')
print()
print(cc['type'].value_counts().to_string())

In [ ]:
# --- Load UMAP projections and merge ---

proj_path = EMBEDDINGS / 'projections.json'
if proj_path.exists():
    with open(proj_path) as f:
        projections = pd.DataFrame(json.load(f))
    all_proj = all_artifacts.merge(projections, on='id', how='inner')
    cc_proj  = cc.merge(projections, on='id', how='inner')
    print(f'Projections loaded: {len(projections)} entries')
else:
    print('projections.json not found — run Step 5 first.')
    all_proj = cc_proj = None

In [ ]:
# --- Semantic space: all artifacts, carbon capture highlighted ---

if all_proj is not None:
    all_proj['highlight'] = all_proj['id'].isin(cc['id'])
    all_proj['label'] = all_proj['highlight'].map(
        {True: 'Carbon capture', False: 'Other'}
    )

    fig = px.scatter(
        all_proj,
        x='x', y='y',
        color='label',
        color_discrete_map={'Carbon capture': '#e15759', 'Other': '#cccccc'},
        opacity=0.6,
        hover_data=['title', 'type', 'year'],
        title='Semantic Space — Carbon Capture Highlighted',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    )
    fig.update_traces(marker_size=4)
    fig.update_layout(legend_title_text='')
    fig.show()

In [ ]:
# --- Semantic space: carbon-capture artifacts only, coloured by type ---

if cc_proj is not None and len(cc_proj) > 0:
    fig = px.scatter(
        cc_proj,
        x='x', y='y',
        color='type',
        hover_data=['title', 'year', 'city'],
        title='Carbon Capture Artifacts — Semantic Space by Type',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    )
    fig.update_traces(marker_size=6)
    fig.show()

In [ ]:
# --- Timeline: carbon capture activity by year and artifact type ---

cc_by_year = (
    cc.groupby(['year', 'type'])
    .size()
    .reset_index(name='count')
)

fig = px.bar(
    cc_by_year,
    x='year', y='count', color='type',
    barmode='stack',
    title='Carbon Capture Activity by Year and Artifact Type',
    labels={'year': 'Year', 'count': 'Number of Artifacts'},
)
fig.show()

In [ ]:
# --- City-level summary ---

cc_cities = (
    cc.groupby(['city', 'country', 'type'])
    .size()
    .reset_index(name='count')
    .pivot_table(index=['city', 'country'], columns='type', values='count', fill_value=0)
    .reset_index()
)
cc_cities['total'] = cc_cities.drop(columns=['city', 'country'], errors='ignore').sum(axis=1)

print('Top cities for carbon capture activity:')
print(cc_cities.sort_values('total', ascending=False).head(15).to_string(index=False))